In [6]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, AutoConfig
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import classification_report
from glassbox_vit.distilling import KnowledgeDistillationTrainer
import torch.nn.functional as F

In [2]:
def evaluate_and_compare_models(teacher_model, student_model, test_loader, device):
    """
    Evaluates both the teacher and the distilled student model on a hold-out test set,
    generating a classification report for each.
    """
    teacher_model.eval()
    student_model.eval()

    all_labels = []
    teacher_preds = []
    student_preds = []

    print("\nEvaluating both models on the test dataset...")

    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            # Teacher predictions
            out_teacher = teacher_model(pixel_values=pixel_values).logits
            preds_t = torch.argmax(out_teacher, dim=-1)

            # Student predictions
            out_student = student_model(pixel_values=pixel_values).logits
            preds_s = torch.argmax(out_student, dim=-1)

            all_labels.extend(labels.cpu().numpy())
            teacher_preds.extend(preds_t.cpu().numpy())
            student_preds.extend(preds_s.cpu().numpy())

    # Target class names for the 'beans' dataset
    target_names = ['angular_leaf_spot', 'bean_rust', 'healthy']

    print("\n" + "="*60)
    print("CLASSIFICATION REPORT: TEACHER MODEL (ViT-Base)")
    print("="*60)
    print(classification_report(all_labels, teacher_preds, target_names=target_names))

    print("\n" + "="*60)
    print("CLASSIFICATION REPORT: DISTILLED STUDENT (ResNet-18)")
    print("="*60)
    print(classification_report(all_labels, student_preds, target_names=target_names))

In [12]:
# Dataset Preparation
print("Loading dataset...")
dataset = load_dataset("AI-Lab-Makerere/beans")

teacher_id = "nateraw/vit-base-beans"
processor = AutoImageProcessor.from_pretrained(teacher_id)

def transform(example_batch):
    inputs = processor([x for x in example_batch['image']], return_tensors='pt')
    inputs['labels'] = example_batch['labels']
    return inputs

prepared_ds = dataset.with_transform(transform)
# Restrict training data to 100 samples for a low-data regime
reduced_train_ds = prepared_ds['train'].shuffle(seed=42).select(range(100))

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

train_loader = DataLoader(reduced_train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(prepared_ds['validation'], batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(prepared_ds['test'], batch_size=16, shuffle=False, collate_fn=collate_fn)

# Model Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

teacher_model = AutoModelForImageClassification.from_pretrained(teacher_id)

student_id = "microsoft/resnet-18"
student_model = AutoModelForImageClassification.from_pretrained(
    student_id,
    num_labels=3,
    ignore_mismatched_sizes=True
)

# Training Configuration
optimizer = AdamW(student_model.parameters(), lr=5e-5)

trainer = KnowledgeDistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    temperature=4.0,
    alpha=0.5
)

# Execution
trainer.train(epochs=5)

# Final Evaluation and Comparison
evaluate_and_compare_models(
    teacher_model=trainer.teacher_model,
    student_model=trainer.student_model,
    test_loader=test_loader,
    device=trainer.device
)

Loading dataset...
Using device: cuda


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([3, 512])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Starting Knowledge Distillation on cuda for 5 epochs...
Epoch [1/5] | Train Loss: 1.3282 | Val Loss: 0.8641 | Val Accuracy: 58.65%
Epoch [2/5] | Train Loss: 0.5211 | Val Loss: 0.5942 | Val Accuracy: 72.93%
Epoch [3/5] | Train Loss: 0.1957 | Val Loss: 0.4814 | Val Accuracy: 81.20%
Epoch [4/5] | Train Loss: 0.2185 | Val Loss: 0.4400 | Val Accuracy: 81.95%
Epoch [5/5] | Train Loss: 0.1197 | Val Loss: 0.4464 | Val Accuracy: 81.95%
Training complete! The student has been successfully distilled.

Evaluating both models on the test dataset...

CLASSIFICATION REPORT: TEACHER MODEL (ViT-Base)
                   precision    recall  f1-score   support

angular_leaf_spot       1.00      0.93      0.96        43
        bean_rust       0.91      1.00      0.96        43
          healthy       1.00      0.98      0.99        42

         accuracy                           0.97       128
        macro avg       0.97      0.97      0.97       128
     weighted avg       0.97      0.97      0.97     

In [15]:
def evaluate_validation(model, val_loader, device):
    """
    Computes validation loss and accuracy for a given model.
    """
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits

            loss = F.cross_entropy(logits, labels)
            running_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            correct_predictions += (preds == labels).sum().item()
            total_samples += labels.size(0)

    return (running_loss / len(val_loader)), (correct_predictions / total_samples)

def train_standard_model(model, train_loader, val_loader, optimizer, device, epochs=10):
    """
    Executes a standard training loop (fine-tuning) without knowledge distillation.
    """
    print(f"\n--- Starting Standard Training on {device} ({epochs} epochs) ---")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            loss = F.cross_entropy(outputs.logits, labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        val_loss, val_acc = evaluate_validation(model, val_loader, device)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

def evaluate_and_compare_models(standard_model, distilled_model, test_loader, device):
    """
    Evaluates both models on the test set and prints their classification reports.
    """
    standard_model.eval()
    distilled_model.eval()

    all_labels = []
    std_preds = []
    kd_preds = []

    print("\nEvaluating both models on the test dataset...")

    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            out_std = standard_model(pixel_values=pixel_values).logits
            out_kd = distilled_model(pixel_values=pixel_values).logits

            std_preds.extend(torch.argmax(out_std, dim=-1).cpu().numpy())
            kd_preds.extend(torch.argmax(out_kd, dim=-1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    target_names = ['angular_leaf_spot', 'bean_rust', 'healthy']

    print("\n" + "="*65)
    print("CLASSIFICATION REPORT: STANDALONE MODEL (Random Init)")
    print("="*65)
    print(classification_report(all_labels, std_preds, target_names=target_names, zero_division=0))

    print("\n" + "="*65)
    print("CLASSIFICATION REPORT: DISTILLED MODEL (Random Init)")
    print("="*65)
    print(classification_report(all_labels, kd_preds, target_names=target_names, zero_division=0))





In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Dataset Preparation
print("Loading and preparing dataset...")
dataset = load_dataset("AI-Lab-Makerere/beans")
teacher_id = "nateraw/vit-base-beans"
processor = AutoImageProcessor.from_pretrained(teacher_id)

def transform(example_batch):
    inputs = processor([x for x in example_batch['image']], return_tensors='pt')
    inputs['labels'] = example_batch['labels']
    return inputs

prepared_ds = dataset.with_transform(transform)

# Restrict training data to 100 samples to simulate low-data regime
reduced_train_ds = prepared_ds['train'].shuffle(seed=42).select(range(100))

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

train_loader = DataLoader(reduced_train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(prepared_ds['validation'], batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(prepared_ds['test'], batch_size=16, shuffle=False, collate_fn=collate_fn)

# Model Initialization
# Teacher model (Pre-trained and fixed)
teacher_model = AutoModelForImageClassification.from_pretrained(teacher_id).to(device)

# Initialize Students from scratch (random weights) instead of pre-trained
student_id = "microsoft/resnet-18"
config = AutoConfig.from_pretrained(student_id, num_labels=3)

student_model_standard = AutoModelForImageClassification.from_config(config).to(device)
student_model_kd = AutoModelForImageClassification.from_config(config).to(device)

# Standard Training Execution
optimizer_std = AdamW(student_model_standard.parameters(), lr=1e-4)
train_standard_model(student_model_standard, train_loader, val_loader, optimizer_std, device, epochs=25)

# Knowledge Distillation Training Execution
optimizer_kd = AdamW(student_model_kd.parameters(), lr=1e-4)
trainer = KnowledgeDistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_model_kd,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_kd,
    device=device,
    temperature=4.0,
    alpha=0.5
)
# Train the distilled model for the exact same number of epochs
trainer.train(epochs=25)

# Final Comparison
evaluate_and_compare_models(student_model_standard, trainer.student_model, test_loader, device)

Using device: cuda
Loading and preparing dataset...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.



--- Starting Standard Training on cuda (25 epochs) ---
Epoch [1/25] | Train Loss: 1.0318 | Val Loss: 1.1898 | Val Acc: 33.83%
Epoch [2/25] | Train Loss: 0.7479 | Val Loss: 1.1524 | Val Acc: 36.84%
Epoch [3/25] | Train Loss: 0.5075 | Val Loss: 1.7444 | Val Acc: 33.08%
Epoch [4/25] | Train Loss: 0.3670 | Val Loss: 1.8651 | Val Acc: 33.08%
Epoch [5/25] | Train Loss: 0.2412 | Val Loss: 1.7365 | Val Acc: 35.34%
Epoch [6/25] | Train Loss: 0.3018 | Val Loss: 1.5964 | Val Acc: 45.11%
Epoch [7/25] | Train Loss: 0.0788 | Val Loss: 1.5682 | Val Acc: 49.62%
Epoch [8/25] | Train Loss: 0.0652 | Val Loss: 1.1698 | Val Acc: 55.64%
Epoch [9/25] | Train Loss: 0.0500 | Val Loss: 1.0722 | Val Acc: 54.89%
Epoch [10/25] | Train Loss: 0.1239 | Val Loss: 1.2493 | Val Acc: 55.64%
Epoch [11/25] | Train Loss: 0.0444 | Val Loss: 1.5779 | Val Acc: 46.62%
Epoch [12/25] | Train Loss: 0.1508 | Val Loss: 1.4277 | Val Acc: 55.64%
Epoch [13/25] | Train Loss: 0.0835 | Val Loss: 1.4560 | Val Acc: 60.90%
Epoch [14/25] | T